In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark= SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
from pyspark.sql.functions import *

In [3]:
loan_rep_df = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/user/itv024771/landing_club/loan_repayment")

In [4]:
loan_rep_df.show()

+--------------------+---------------+---------+------------+--------+---------------+------------+---------+----------------+---------------+-------------+------------------+-----------+
|              mem_id|funded_amnt_inv|loan_amnt|tot_coll_amt|int_rate|last_pymnt_amnt|last_pymnt_d|     term|     total_pymnt|total_rec_prncp|total_rec_int|total_rec_late_fee|installment|
+--------------------+---------------+---------+------------+--------+---------------+------------+---------+----------------+---------------+-------------+------------------+-----------+
|e590be3a0a8891e6a...|        40000.0|    40000|           0|    8.46|          819.9|    Feb-2019|60 months|          3242.0|        2174.46|      1067.54|               0.0|      819.9|
|84f70937bf62e697c...|        10000.0|    10000|           0|   11.55|          330.0|    Feb-2019|36 months|         1313.58|         948.59|       364.99|               0.0|      330.0|
|bef1d87f7e254abf8...|        10000.0|    10000|           0

In [5]:
loan_rep_df.printSchema()

root
 |-- mem_id: string (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- tot_coll_amt: integer (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- last_pymnt_amnt: double (nullable = true)
 |-- last_pymnt_d: string (nullable = true)
 |-- term: string (nullable = true)
 |-- total_pymnt: double (nullable = true)
 |-- total_rec_prncp: double (nullable = true)
 |-- total_rec_int: double (nullable = true)
 |-- total_rec_late_fee: double (nullable = true)
 |-- installment: double (nullable = true)



In [6]:
ingest_loan_rep_df = loan_rep_df.withColumn("ingest_date", current_timestamp())

In [7]:
ingest_loan_rep_df.show()

+--------------------+---------------+---------+------------+--------+---------------+------------+---------+----------------+---------------+-------------+------------------+-----------+--------------------+
|              mem_id|funded_amnt_inv|loan_amnt|tot_coll_amt|int_rate|last_pymnt_amnt|last_pymnt_d|     term|     total_pymnt|total_rec_prncp|total_rec_int|total_rec_late_fee|installment|         ingest_date|
+--------------------+---------------+---------+------------+--------+---------------+------------+---------+----------------+---------------+-------------+------------------+-----------+--------------------+
|e590be3a0a8891e6a...|        40000.0|    40000|           0|    8.46|          819.9|    Feb-2019|60 months|          3242.0|        2174.46|      1067.54|               0.0|      819.9|2026-03-28 09:55:...|
|84f70937bf62e697c...|        10000.0|    10000|           0|   11.55|          330.0|    Feb-2019|36 months|         1313.58|         948.59|       364.99|        

In [8]:
ingest_loan_rep_df.createOrReplaceTempView("loan_rep")

In [9]:
spark.sql("select * from loan_rep")

mem_id,funded_amnt_inv,loan_amnt,tot_coll_amt,int_rate,last_pymnt_amnt,last_pymnt_d,term,total_pymnt,total_rec_prncp,total_rec_int,total_rec_late_fee,installment,ingest_date
e590be3a0a8891e6a...,40000.0,40000,0,8.46,819.9,Feb-2019,60 months,3242.0,2174.46,1067.54,0.0,819.9,2026-03-28 09:55:...
84f70937bf62e697c...,10000.0,10000,0,11.55,330.0,Feb-2019,36 months,1313.58,948.59,364.99,0.0,330.0,2026-03-28 09:55:...
bef1d87f7e254abf8...,10000.0,10000,0,23.4,389.19,Feb-2019,36 months,1543.76,799.78,743.98,0.0,389.19,2026-03-28 09:55:...
527227dcaa7199f1d...,17400.0,17400,0,23.4,17427.42,Dec-2018,36 months,18081.9905258844,17400.0,681.99,0.0,677.19,2026-03-28 09:55:...
106a97c1290320bd0...,9200.0,9200,0,10.47,298.9,Feb-2019,36 months,1190.25,886.03,304.22,0.0,298.9,2026-03-28 09:55:...
cca52240a7f8cff6a...,13500.0,13500,0,7.21,418.14,Feb-2019,36 months,1707.71,1360.31,347.4,0.0,418.14,2026-03-28 09:55:...
61c58a97c9a240a77...,18000.0,18000,0,6.11,548.5,Feb-2019,36 months,2187.89,1841.4,346.49,0.0,548.5,2026-03-28 09:55:...
285603e783c33ccd5...,11000.0,11000,0,7.21,340.71,Feb-2019,36 months,1358.43,1108.41,250.02,0.0,340.71,2026-03-28 09:55:...
dec89910e65ef8ac2...,6000.0,6000,0,12.73,201.39,Feb-2019,36 months,801.32,559.79,241.53,0.0,201.39,2026-03-28 09:55:...
2a563aa7f36c18b33...,15000.0,15000,0,14.47,352.69,Feb-2019,60 months,1398.7,699.79,698.91,0.0,352.69,2026-03-28 09:55:...


In [10]:
spark.sql("select count(*) from loan_rep where total_rec_prncp is null")

count(1)
1


In [11]:
col_to_chk = ["total_pymnt",'total_rec_prncp','total_rec_int','last_pymnt_amnt']

In [12]:
loan_rep_filtered_df = ingest_loan_rep_df.na.drop(subset=col_to_chk)

In [13]:
loan_rep_filtered_df.count()

220996

In [14]:
loan_rep_filtered_df.createOrReplaceTempView("loan_rep")

In [15]:
spark.sql("select count(*) from loan_rep where total_pymnt ==0 and total_rec_prncp != 0")

count(1)
0


In [16]:
spark.sql("select count(*) from loan_rep where total_rec_prncp == 0.0 and total_pymnt ==  0.0")

count(1)
262


In [17]:
spark.sql("select count(*) from loan_rep where total_rec_prncp != 0.0 and total_pymnt !=  0.0")

count(1)
220722


In [18]:
loan_repayment_df = loan_rep_filtered_df.filter("total_rec_prncp != 0.0 and total_pymnt !=  0.0")

In [19]:
loan_repayment_df.count()

220722

In [20]:
loan_repayment_df.show()

+--------------------+---------------+---------+------------+--------+---------------+------------+---------+----------------+---------------+-------------+------------------+-----------+--------------------+
|              mem_id|funded_amnt_inv|loan_amnt|tot_coll_amt|int_rate|last_pymnt_amnt|last_pymnt_d|     term|     total_pymnt|total_rec_prncp|total_rec_int|total_rec_late_fee|installment|         ingest_date|
+--------------------+---------------+---------+------------+--------+---------------+------------+---------+----------------+---------------+-------------+------------------+-----------+--------------------+
|e590be3a0a8891e6a...|        40000.0|    40000|           0|    8.46|          819.9|    Feb-2019|60 months|          3242.0|        2174.46|      1067.54|               0.0|      819.9|2026-03-28 09:55:...|
|84f70937bf62e697c...|        10000.0|    10000|           0|   11.55|          330.0|    Feb-2019|36 months|         1313.58|         948.59|       364.99|        

In [21]:
loan_repayment_df.createOrReplaceTempView("loan_rep")

In [24]:
spark.sql("select * from loan_rep where last_pymnt_d is null")

mem_id,funded_amnt_inv,loan_amnt,tot_coll_amt,int_rate,last_pymnt_amnt,last_pymnt_d,term,total_pymnt,total_rec_prncp,total_rec_int,total_rec_late_fee,installment,ingest_date
ae9444173bf0fd852...,18000.0,18000,6693,24.37,0.0,null,36 months,648.78,344.15,304.63,0.0,709.7,2026-03-28 09:55:...


In [27]:
loan_repayment_df.repartition(1).write \
.format("parquet") \
.mode("overwrite") \
.option("header",True) \
.option("path","/user/itv024771/landing_club/cleaned/loan_repayment") \
.save()